In [ ]:
import pandas as pd
import numpy as np

import matplotlib
from matplotlib import colormaps
from matplotlib.cm import get_cmap
%matplotlib inline
  # non-interactive, avoids GUI conflicts
import matplotlib.pyplot as plt
import scanpy as sc
import scanpy.external as sce
import os 
import matplotlib.pyplot as plt
import scipy.sparse as sp
from scipy.io import mmwrite, mmread

import sys
bnsi_path = "/scicore/home/nimwegen/degroo0000/Bonsai-data-representation/"
sys.path.append(bnsi_path)

from paper_figure_scripts_and_notebooks.simulating_datasets.analyzing_simulated_datasets.knn_recall_helpers import do_pca, do_logp1, fit_DTNE


In [ ]:
results_dir = "/scicore/home/nimwegen/GROUP/Projects/bonsai_runs/paper_figures_datasets/hao_satija_2021/"

raw_cnts_dir = os.path.join(results_dir, "UMI")


annot_dir = os.path.join(results_dir, "annotation", "cell_annot_sub.csv")
counts_file = os.path.join(raw_cnts_dir, "prom_expr_matrix-SUB.mtx")
genes_file = os.path.join(raw_cnts_dir, "prom_expr_promoters.tsv")
print(counts_file)


annot_df = pd.read_csv(annot_dir)

genes_df = pd.read_csv(genes_file, header=None, names=["geneName"])                      
gene_expression = mmread(counts_file).toarray()

display(annot_df.head())
display(genes_df.head())

gene_expression.shape, annot_df.shape

In [ ]:
print(annot_df.rna_annotations.unique())
rna_celltypes = [
    "B", 
    "CD8 T",
    "CD14+ Mono", 
    "CD16+ Mono", 
    "CD34+",
    "DC", 
    "Eryth", 
    "Memory CD4 T", 
    "Mk", 
    "NK",
    "Naive CD4 T", 
    "T/Mono doublets", 
    "pDCs"
                ]

cmap = get_cmap("tab20")
colors = [cmap(i / 20) for i in range(len(rna_celltypes))]  

rna_col_dict = dict(zip(rna_celltypes, colors))
rna_col_dict

In [ ]:
adata = sc.AnnData(gene_expression.T)   # transpose so cells are rows
adata.obs = annot_df
adata.obs['rna_annotations'] = adata.obs['rna_annotations'].astype('category')

In [ ]:
adata.var_names = genes_df.geneName
adata.var_names_make_unique()

adata.obs_names = adata.obs_names.astype(str)
adata.var_names = adata.var_names.astype(str)

adata.obs_names_make_unique()
adata.var_names_make_unique()

In [ ]:
sc.pp.filter_genes(adata, min_cells=10)
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_cells(adata, min_counts=200)

In [ ]:
# Normalizing to median total counts
sc.pp.normalize_total(adata, exclude_highly_expressed=False)

# 6) Preprocessing: normalize to total count = 1e6 and log1p
sc.pp.log1p(adata)
print("Normalized to total_count=1e6 and log1p applied.")

In [ ]:
# feature selection
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
# dimensionality reduction
sc.pp.pca(adata)
# nearest neighbor graph construction
#  compute the neighborhood graph of cells using the PCA representation of the data matrix.
sc.pp.neighbors(adata) # default 15 neighbours, 
# This graph can then be embedded in two dimensions for visualiztion with UMAP
sc.tl.umap(adata)# (adata, min_dist=0.5, spread=1.0, n_components=2, maxiter=None, alpha=1.0, gamma=1.0, negative_sample_rate=5, init_pos='spectral', random_state=0, a=None, b=None, copy=False, method='umap', neighbors_key=None)

In [ ]:
adata

In [ ]:
N_PCS_DTNE = 500

# Perform logp1
logp1 = do_logp1(gene_expression)
print("logp1 done")

# Perform PCA 
pca_projected = do_pca(logp1, n_comps_list=[N_PCS_DTNE])
print("pca_projected done")

# Perform DTNE
dtne_fit = fit_DTNE(pca_projected[N_PCS_DTNE])
print("dtne_fit done")

In [ ]:
# Final Figure

from matplotlib.lines import Line2D

fig, axes = plt.subplots(2,2, figsize=(7,5), dpi=300)

sc.pl.pca(adata,
           color=['rna_annotations'], 
           palette=[rna_col_dict[c] for c in adata.obs['rna_annotations'].cat.categories],
            frameon=False,
           title = "PCA",
          legend_loc=None,
           show=False, ax=axes[0,0])


sc.pl.umap(adata,
           color=['rna_annotations'], 
           palette=[rna_col_dict[c] for c in adata.obs['rna_annotations'].cat.categories],
            frameon=False,
           title = "UMAP",
           legend_loc=None,
           show=False, ax=axes[0,1])

sce.tl.phate(adata, k=10, a=20, t='auto')
sce.pl.phate(adata,
           color=['rna_annotations'], 
           palette=[rna_col_dict[c] for c in adata.obs['rna_annotations'].cat.categories],
            frameon=False,
           title = "PHATE",
            legend_loc=None,
           show=False, ax=axes[1,0])

axes[1,1].scatter(x=dtne_fit[0,:],
                  y=dtne_fit[1,:],
                  color=[rna_col_dict[x] for x in annot_df["rna_annotations"]],
                  s=2,
                 )
axes[1,1].axis('off')
axes[1,1].set_title("DTNE")

cats =  list(adata.obs["rna_annotations"].cat.categories)
handles = [
    Line2D([0], [0],
           marker='o',
           linestyle='',
           markerfacecolor=rna_col_dict[c],
           markeredgecolor='none',
           markersize=6,
           label=c)
    for c in cats
]

fig.legend(
    handles=handles,
    labels=cats,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.2),
    ncol=4,
    fancybox=True,
    shadow=False,
    facecolor="#e5e5e5"
)

plt.tight_layout() 
plt.show()